# Data loader & preprocessing


Cell 1 — Install dependencies



In [ ]:
!pip install -q torch torch-geometric pyarrow pandas numpy scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.3 MB/s eta 0:00:00


Cell 2 — Mount Drive and verify your file exists

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

DRIVE_FOLDER = '/content/drive/MyDrive/traffic_project'
PARQUET_PATH = f'{DRIVE_FOLDER}/merged_dataset_full.parquet'

# Verify the file is there and check its size
size_gb = os.path.getsize(PARQUET_PATH) / 1024**3
print(f"✅ Found dataset: {size_gb:.2f} GB")

Mounted at /content/drive
✅ Found dataset: 0.07 GB


Cell 3 — Load and inspect the data

In [ ]:
# Load the full parquet file
# This takes ~30-60 seconds and uses ~2.5 GB of RAM
print("Loading dataset... (30-60 seconds)")
df = pd.read_parquet(PARQUET_PATH)

print(f"\n✅ Loaded successfully")
print(f"   Shape: {df.shape}")
print(f"   Columns: {list(df.columns)}")
print(f"\nFirst 3 rows:")
print(df.head(3))
print(f"\nData types:")
print(df.dtypes)

Loading dataset... (30-60 seconds)

✅ Loaded successfully
   Shape: (16937700, 16)
   Columns: ['timestamp', 'sensor_id', 'Latitude', 'Longitude', 'speed', 'hour', 'dayofweek', 'is_weekend', 'Temperature(F)', 'Humidity(%)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Weather_Condition', 'acc_count_60min', 'acc_max_severity', 'acc_mins_since']

First 3 rows:
            timestamp  sensor_id   Latitude   Longitude  speed  hour  \
0 2017-01-01 00:00:00     400001  37.364085 -121.901149   71.4     0   
1 2017-01-01 00:05:00     400001  37.364085 -121.901149   71.6     0   
2 2017-01-01 00:10:00     400001  37.364085 -121.901149   71.6     0   

   dayofweek  is_weekend  Temperature(F)  Humidity(%)  Visibility(mi)  \
0          6           1            44.1         79.0            10.0   
1          6           1            44.1         79.0            10.0   
2          6           1            44.1         79.0            10.0   

   Wind_Speed(mph) Weather_Condition  acc_count_60min  acc_max_s

Cell 4 — Sort and understand the structure

In [ ]:
# CRITICAL: sort by sensor then time
# Your data must be ordered: all timesteps for sensor A, then all for sensor B, etc.
df = df.sort_values(['sensor_id', 'timestamp']).reset_index(drop=True)

# Quick sanity check
print("Unique sensors:", df['sensor_id'].nunique())
print("Unique timestamps:", df['timestamp'].nunique())
print("Timesteps per sensor (should all be ~52,116):")
print(df.groupby('sensor_id').size().describe())

Unique sensors: 325
Unique timestamps: 52116
Timesteps per sensor (should all be ~52,116):
count      325.0
mean     52116.0
std          0.0
min      52116.0
25%      52116.0
50%      52116.0
75%      52116.0
max      52116.0
dtype: float64


Cell 5 — Encode the weather category

In [ ]:
# Weather_Condition is text ("Clear", "Rain", etc.)
# Models need numbers, not text
# We convert it to a numeric code

df['Weather_Condition'] = df['Weather_Condition'].fillna('Unknown')

# Map unique weather conditions to integers
weather_categories = df['Weather_Condition'].unique()
weather_map = {w: i for i, w in enumerate(sorted(weather_categories))}
df['weather_code'] = df['Weather_Condition'].map(weather_map)

print(f"Weather categories ({len(weather_map)} total):")
for k, v in sorted(weather_map.items(), key=lambda x: x[1]):
    count = (df['Weather_Condition'] == k).sum()
    print(f"  {v}: {k} ({count:,} rows)")

# Save the mapping so you can decode predictions later
import json
with open(f'{DRIVE_FOLDER}/weather_map.json', 'w') as f:
    json.dump(weather_map, f)
print("\n✅ Weather map saved to Drive")

Weather categories (18 total):
  0: Clear (6,645,600 rows)
  1: Fog (28,600 rows)
  2: Haze (118,950 rows)
  3: Heavy Rain (50,050 rows)
  4: Light Drizzle (15,275 rows)
  5: Light Rain (1,154,725 rows)
  6: Light Thunderstorms and Rain (6,175 rows)
  7: Mist (11,700 rows)
  8: Mostly Cloudy (2,445,625 rows)
  9: Overcast (2,605,525 rows)
  10: Partly Cloudy (1,958,775 rows)
  11: Patches of Fog (1,300 rows)
  12: Rain (237,575 rows)
  13: Rain Showers (8,125 rows)
  14: Scattered Clouds (1,266,200 rows)
  15: Shallow Fog (4,550 rows)
  16: Squalls (975 rows)
  17: Unknown (377,975 rows)

✅ Weather map saved to Drive


Cell 6 — Handle missing values

In [ ]:
# Fill the ~2% missing weather values with the column median
# This is the simplest and most robust approach for missing data

weather_numeric_cols = [
    'Temperature(F)', 'Humidity(%)', 'Visibility(mi)', 'Wind_Speed(mph)'
]

print("Missing values before filling:")
for col in weather_numeric_cols:
    pct = df[col].isna().mean() * 100
    print(f"  {col}: {pct:.2f}%")

for col in weather_numeric_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

print("\nMissing values after filling:")
for col in weather_numeric_cols:
    print(f"  {col}: {df[col].isna().mean()*100:.2f}%")

Missing values before filling:
  Temperature(F): 2.19%
  Humidity(%): 2.23%
  Visibility(mi): 2.22%
  Wind_Speed(mph): 18.67%

Missing values after filling:
  Temperature(F): 0.00%
  Humidity(%): 0.00%
  Visibility(mi): 0.00%
  Wind_Speed(mph): 0.00%


Cell 7 — Define your feature columns

In [ ]:
# These are the features your model will SEE (inputs)
FEATURE_COLS = [
    'speed',                    # traffic speed (also the target)
    'hour',                     # 0-23
    'dayofweek',                # 0-6
    'is_weekend',               # 0 or 1
    'Temperature(F)',           # weather
    'Humidity(%)',              # weather
    'Visibility(mi)',           # weather
    'Wind_Speed(mph)',          # weather
    'weather_code',             # weather condition (encoded)
    'acc_count_60min',          # accident context
    'acc_max_severity',         # accident context
    'acc_mins_since',           # accident context
]

TARGET_COL = 'speed'           # what the model PREDICTS

print(f"Number of input features (F): {len(FEATURE_COLS)}")
print(f"Target: {TARGET_COL}")
print(f"\nFeature columns: {FEATURE_COLS}")

Number of input features (F): 12
Target: speed

Feature columns: ['speed', 'hour', 'dayofweek', 'is_weekend', 'Temperature(F)', 'Humidity(%)', 'Visibility(mi)', 'Wind_Speed(mph)', 'weather_code', 'acc_count_60min', 'acc_max_severity', 'acc_mins_since']


Cell 8 — Normalize the features

In [ ]:
# ⚠️ IMPORTANT: Normalization is one of the most critical steps in ML
# Without it, a feature with range 0-85 (speed) overwhelms one with range 0-1 (is_weekend)
# We scale everything to roughly 0-1 using the TRAINING data only
# (never look at validation/test data during normalization — that's data leakage)

import numpy as np

# Chronological split FIRST, then normalize
# PEMS-BAY has 52,116 timesteps
# 70% train = timesteps 0 to 36,481
# 10% val   = timesteps 36,481 to 41,693
# 20% test  = timesteps 41,693 to 52,116

all_timestamps = sorted(df['timestamp'].unique())
n = len(all_timestamps)

train_end_idx = int(n * 0.70)
val_end_idx   = int(n * 0.80)

train_cutoff = all_timestamps[train_end_idx]
val_cutoff   = all_timestamps[val_end_idx]

print(f"Total unique timestamps: {n:,}")
print(f"\nSplit:")
print(f"  Train: {all_timestamps[0]}  →  {train_cutoff}  ({train_end_idx:,} steps)")
print(f"  Val:   {train_cutoff}  →  {val_cutoff}  ({val_end_idx - train_end_idx:,} steps)")
print(f"  Test:  {val_cutoff}  →  {all_timestamps[-1]}  ({n - val_end_idx:,} steps)")

Total unique timestamps: 52,116

Split:
  Train: 2017-01-01 00:00:00  →  2017-05-07 17:05:00  (36,481 steps)
  Val:   2017-05-07 17:05:00  →  2017-05-25 19:20:00  (5,211 steps)
  Test:  2017-05-25 19:20:00  →  2017-06-30 23:55:00  (10,424 steps)


Cell 9 — Compute normalization statistics from training data only

In [ ]:
from sklearn.preprocessing import StandardScaler
import pickle

# Separate training rows
train_mask = df['timestamp'] <= train_cutoff

# Fit scaler ONLY on training data
scaler = StandardScaler()
scaler.fit(df[train_mask][FEATURE_COLS])

# Apply to ALL data (transform, not fit)
df_scaled = df.copy()
df_scaled[FEATURE_COLS] = scaler.transform(df[FEATURE_COLS])

print("✅ Normalization complete")
print(f"\nPost-scaling stats (training set):")
print(df_scaled[train_mask][FEATURE_COLS].describe().round(3))

# Save the scaler — you'll need it to decode predictions back to mph
with open(f'{DRIVE_FOLDER}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print(f"\n✅ Scaler saved to Drive")

✅ Normalization complete

Post-scaling stats (training set):
              speed          hour     dayofweek    is_weekend  Temperature(F)  \
count  1.185665e+07  1.185665e+07  1.185665e+07  1.185665e+07    1.185665e+07   
mean   0.000000e+00 -0.000000e+00  0.000000e+00 -0.000000e+00   -0.000000e+00   
std    1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00    1.000000e+00   
min   -6.648000e+00 -1.660000e+00 -1.503000e+00 -6.380000e-01   -3.212000e+00   
25%   -5.700000e-02 -9.370000e-01 -1.005000e+00 -6.380000e-01   -5.990000e-01   
50%    2.720000e-01 -7.000000e-02 -8.000000e-03 -6.380000e-01   -5.000000e-03   
75%    5.050000e-01  7.980000e-01  9.890000e-01  1.567000e+00    5.290000e-01   
max    2.370000e+00  1.665000e+00  1.487000e+00  1.567000e+00    4.615000e+00   

        Humidity(%)  Visibility(mi)  Wind_Speed(mph)  weather_code  \
count  1.185665e+07    1.185665e+07     1.185665e+07  1.185665e+07   
mean  -0.000000e+00   -0.000000e+00    -0.000000e+00 -0.000000e+00   

Cell 10 — Reshape into sensor × time matrix

In [ ]:
import numpy as np

print("Reshaping to 3D array using vectorized operations...")
print("(Should take 2-5 minutes, not hours)\n")

# Get ordered lists
sensors    = sorted(df_scaled['sensor_id'].unique())
timestamps = sorted(df_scaled['timestamp'].unique())
n_sensors  = len(sensors)
n_times    = len(timestamps)
n_features = len(FEATURE_COLS)

print(f"Sensors:    {n_sensors}")
print(f"Timesteps:  {n_times:,}")
print(f"Features:   {n_features}")
print(f"Target shape: ({n_times}, {n_sensors}, {n_features})")

# Build index maps (sensor_id → column index, timestamp → row index)
sensor_to_idx = {s: i for i, s in enumerate(sensors)}
time_to_idx   = {t: i for i, t in enumerate(timestamps)}

# Map sensor IDs and timestamps to integer indices (vectorized)
print("\nBuilding index arrays...")
sensor_indices = df_scaled['sensor_id'].map(sensor_to_idx).values  # (16.9M,)
time_indices   = df_scaled['timestamp'].map(time_to_idx).values     # (16.9M,)

# Extract feature values as a numpy array (16.9M × 12)
print("Extracting feature values...")
feature_values = df_scaled[FEATURE_COLS].values.astype(np.float32)  # (16.9M, 12)

# Pre-allocate the 3D array
print("Allocating 3D array...")
data_3d = np.zeros((n_times, n_sensors, n_features), dtype=np.float32)

# Fill using vectorized fancy indexing — no Python loop at all
print("Filling 3D array (vectorized)...")
data_3d[time_indices, sensor_indices, :] = feature_values

print(f"\n✅ 3D array built: {data_3d.shape}")
print(f"   Memory: {data_3d.nbytes / 1024**2:.1f} MB")

# Verify — check one sensor's speed values make sense
sample_sensor_idx = 0
speeds_sample = data_3d[:5, sample_sensor_idx, 0]
print(f"\nSanity check (first 5 timesteps, sensor 0, speed column):")
print(f"  Scaled values: {speeds_sample}")
print(f"  (Should be non-zero numbers around -3 to +3)")

# Save to Drive
print("\nSaving to Drive...")
np.save(f'{DRIVE_FOLDER}/data_3d.npy', data_3d)

# Save index maps for later use
with open(f'{DRIVE_FOLDER}/sensors_list.json', 'w') as f:
    json.dump([str(s) for s in sensors], f)
with open(f'{DRIVE_FOLDER}/timestamps_list.json', 'w') as f:
    json.dump([str(t) for t in timestamps], f)

print(f"✅ Saved data_3d.npy to Drive")
print(f"✅ Saved sensors_list.json and timestamps_list.json")


Reshaping to 3D array using vectorized operations...
(Should take 2-5 minutes, not hours)

Sensors:    325
Timesteps:  52,116
Features:   12
Target shape: (52116, 325, 12)

Building index arrays...
Extracting feature values...
Allocating 3D array...
Filling 3D array (vectorized)...

✅ 3D array built: (52116, 325, 12)
   Memory: 775.3 MB

Sanity check (first 5 timesteps, sensor 0, speed column):
  Scaled values: [0.91801184 0.9392055  0.9392055  0.8862213  0.9498024 ]
  (Should be non-zero numbers around -3 to +3)

Saving to Drive...
✅ Saved data_3d.npy to Drive
✅ Saved sensors_list.json and timestamps_list.json

✅ Cell 10 complete — ready for Cell 11


Cell 11 — Build the PyTorch Dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# A PyTorch Dataset is just a class that answers two questions:
# 1. How many samples do I have? (__len__)
# 2. What does sample #i look like? (__getitem__)

class TrafficDataset(Dataset):
    def __init__(self, data_3d, split_indices, input_steps=12, pred_steps=12):
        """
        data_3d:        numpy array of shape (T, N, F)
        split_indices:  which time indices belong to this split (train/val/test)
        input_steps:    how many past timesteps to look at (12 = 1 hour)
        pred_steps:     how many future timesteps to predict (12 = 1 hour ahead)
        """
        self.data = data_3d
        self.indices = split_indices
        self.input_steps = input_steps
        self.pred_steps = pred_steps
        # Speed is the first column in FEATURE_COLS
        self.speed_col_idx = 0

    def __len__(self):
        # Each valid sample needs `input_steps` history + `pred_steps` future
        return len(self.indices)

    def __getitem__(self, i):
        t = self.indices[i]

        # INPUT: past 12 timesteps, all sensors, all features
        # Shape: (input_steps, N, F) = (12, 325, 12)
        x = self.data[t - self.input_steps : t]

        # TARGET: future speed only (we only predict speed, not weather)
        # Shape: (pred_steps, N) = (12, 325)
        y = self.data[t : t + self.pred_steps, :, self.speed_col_idx]

        return torch.tensor(x, dtype=torch.float32), \
               torch.tensor(y, dtype=torch.float32)


# Build valid indices for each split
# A valid index t means: we have t-12 history AND t+12 future
INPUT_STEPS = 12   # 1 hour of history
PRED_STEPS  = 12   # predict up to 1 hour ahead

train_end_t = int(n_times * 0.70)
val_end_t   = int(n_times * 0.80)

# Valid indices: must have enough history AND future within the same split
train_indices = list(range(INPUT_STEPS, train_end_t - PRED_STEPS))
val_indices   = list(range(train_end_t + INPUT_STEPS, val_end_t - PRED_STEPS))
test_indices  = list(range(val_end_t + INPUT_STEPS, n_times - PRED_STEPS))

print(f"Samples per split:")
print(f"  Train: {len(train_indices):,}")
print(f"  Val:   {len(val_indices):,}")
print(f"  Test:  {len(test_indices):,}")

# Create Dataset objects
train_dataset = TrafficDataset(data_3d, train_indices, INPUT_STEPS, PRED_STEPS)
val_dataset   = TrafficDataset(data_3d, val_indices,   INPUT_STEPS, PRED_STEPS)
test_dataset  = TrafficDataset(data_3d, test_indices,  INPUT_STEPS, PRED_STEPS)

# Create DataLoaders (batches data, shuffles train set)
BATCH_SIZE = 32   # safe for Colab free tier RAM

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"\n✅ DataLoaders ready")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train batches: {len(train_loader):,}")

Samples per split:
  Train: 36,457
  Val:   5,187
  Test:  10,400

✅ DataLoaders ready
   Batch size: 32
   Train batches: 1,140


Cell 12 — Verify one batch looks correct

In [ ]:
# Grab one batch and check its shape
x_batch, y_batch = next(iter(train_loader))

print("One batch shapes:")
print(f"  Input  x: {x_batch.shape}  → (batch_size, input_steps, n_sensors, n_features)")
print(f"  Target y: {y_batch.shape}  → (batch_size, pred_steps, n_sensors)")
print(f"\nExpected: x = (32, 12, 325, 12)  |  y = (32, 12, 325)")
print(f"\nInput values (first sample, first timestep, first 5 sensors):")
print(x_batch[0, 0, :5, :])
print(f"\nTarget values (first sample, first timestep, first 5 sensors):")
print(y_batch[0, 0, :5])
print(f"\nValue range check (should be roughly -3 to +3 after normalization):")
print(f"  x min: {x_batch.min():.2f}   x max: {x_batch.max():.2f}")
print(f"  y min: {y_batch.min():.2f}   y max: {y_batch.max():.2f}")

One batch shapes:
  Input  x: torch.Size([32, 12, 325, 12])  → (batch_size, input_steps, n_sensors, n_features)
  Target y: torch.Size([32, 12, 325])  → (batch_size, pred_steps, n_sensors)

Expected: x = (32, 12, 325, 12)  |  y = (32, 12, 325)

Input values (first sample, first timestep, first 5 sensors):
tensor([[ 0.8862, -1.5158,  1.4871,  1.5666, -0.5992,  0.2244,  0.3821,  0.3408,
         -1.2015, -0.3449, -0.3690,  0.3254],
        [ 0.2186, -1.5158,  1.4871,  1.5666, -0.5992,  0.2244,  0.3821,  0.3408,
         -1.2015, -0.3449, -0.3690,  0.3254],
        [ 0.4200, -1.5158,  1.4871,  1.5666, -0.5992,  0.2244,  0.3821,  0.3408,
         -1.2015, -0.3449, -0.3690,  0.3254],
        [ 0.6319, -1.5158,  1.4871,  1.5666, -0.5992,  0.2244,  0.3821,  0.3408,
         -1.2015, -0.3449, -0.3690,  0.3254],
        [ 0.8014, -1.5158,  1.4871,  1.5666, -0.5992,  0.2244,  0.3821,  0.3408,
         -1.2015, -0.3449, -0.3690,  0.3254]])

Target values (first sample, first timestep, first 5 sen

Cell 13 — Load the adjacency matrix

In [ ]:
# The adjacency matrix tells the GNN which sensors are neighbors
# Shape: (325, 325) — entry [i,j] = how strongly sensor i influences sensor j

import pickle

with open(f'{DRIVE_FOLDER}/adj_mx_bay.pkl', 'rb') as f:
    adj_data = pickle.load(f, encoding='latin1')

# adj_data is a list: [sensor_ids_dict, sensor_id_to_idx, adjacency_matrix]
adj_matrix = adj_data[2]   # the actual 325x325 numpy array

print(f"Adjacency matrix shape: {adj_matrix.shape}")
print(f"Non-zero entries: {(adj_matrix > 0).sum():,}")
print(f"Density: {(adj_matrix > 0).mean()*100:.2f}%")
print(f"Avg neighbors per sensor: {(adj_matrix > 0).sum() / adj_matrix.shape[0]:.1f}")
print(f"\nValue range: {adj_matrix.min():.4f} to {adj_matrix.max():.4f}")

# Convert to PyTorch tensor — needed for GNN later
adj_tensor = torch.tensor(adj_matrix, dtype=torch.float32)
print(f"\n✅ Adjacency matrix ready as PyTorch tensor: {adj_tensor.shape}")

# Save for reuse
torch.save(adj_tensor, f'{DRIVE_FOLDER}/adj_tensor.pt')
print(f"✅ Saved adj_tensor.pt to Drive")

Adjacency matrix shape: (325, 325)
Non-zero entries: 2,694
Density: 2.55%
Avg neighbors per sensor: 8.3

Value range: 0.0000 to 1.0000

✅ Adjacency matrix ready as PyTorch tensor: torch.Size([325, 325])
✅ Saved adj_tensor.pt to Drive


Cell 14 — Final summary and save all config

In [ ]:
# Save all configuration variables to a JSON file
# This way every future notebook knows the exact settings used here

config = {
    'n_sensors':    n_sensors,       # 325
    'n_timesteps':  n_times,         # 52,116
    'n_features':   n_features,      # 12
    'feature_cols': FEATURE_COLS,
    'target_col':   TARGET_COL,
    'input_steps':  INPUT_STEPS,     # 12
    'pred_steps':   PRED_STEPS,      # 12
    'batch_size':   BATCH_SIZE,      # 32
    'train_end_t':  train_end_t,
    'val_end_t':    val_end_t,
    'train_samples': len(train_indices),
    'val_samples':   len(val_indices),
    'test_samples':  len(test_indices),
}

with open(f'{DRIVE_FOLDER}/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("="*50)
print("✅ DATA LOADER COMPLETE")
print("="*50)
print(f"\nFiles saved to Drive:")
print(f"  data_3d.npy       — 3D data array (T × N × F)")
print(f"  adj_tensor.pt     — adjacency matrix")
print(f"  scaler.pkl        — normalization scaler")
print(f"  weather_map.json  — weather category encoding")
print(f"  sensors_list.json — ordered sensor IDs")
print(f"  config.json       — all configuration settings")
print(f"\nReady for: Step 2 (Baseline Models)")

✅ DATA LOADER COMPLETE

Files saved to Drive:
  data_3d.npy       — 3D data array (T × N × F)
  adj_tensor.pt     — adjacency matrix
  scaler.pkl        — normalization scaler
  weather_map.json  — weather category encoding
  sensors_list.json — ordered sensor IDs
  config.json       — all configuration settings

Ready for: Step 2 (Baseline Models)


# Baseline models

Cell 1 — Mount Drive and load everything

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import json, pickle, os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

DRIVE_FOLDER = '/content/drive/MyDrive/traffic_project'

# Load config
with open(f'{DRIVE_FOLDER}/config.json') as f:
    config = json.load(f)

# Load 3D data array
print("Loading data_3d.npy...")
data_3d = np.load(f'{DRIVE_FOLDER}/data_3d.npy')
print(f"✅ data_3d shape: {data_3d.shape}")  # (52116, 325, 12)

# Load scaler (to convert predictions back to mph)
with open(f'{DRIVE_FOLDER}/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# Load adjacency matrix
adj_tensor = torch.load(f'{DRIVE_FOLDER}/adj_tensor.pt')
print(f"✅ adj_tensor shape: {adj_tensor.shape}")

# Key config values
N_SENSORS   = config['n_sensors']       # 325
N_TIMES     = config['n_timesteps']     # 52116
N_FEATURES  = config['n_features']     # 12
INPUT_STEPS = config['input_steps']    # 12
PRED_STEPS  = config['pred_steps']     # 12
BATCH_SIZE  = config['batch_size']     # 32
TRAIN_END   = config['train_end_t']    # ~36481
VAL_END     = config['val_end_t']      # ~41693
SPEED_IDX   = 0                        # speed is feature index 0

print(f"\n✅ Config loaded")
print(f"   Sensors: {N_SENSORS}, Timesteps: {N_TIMES}, Features: {N_FEATURES}")
print(f"   Train ends at timestep: {TRAIN_END}")
print(f"   Val ends at timestep:   {VAL_END}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading data_3d.npy...
✅ data_3d shape: (52116, 325, 12)
✅ adj_tensor shape: torch.Size([325, 325])

✅ Config loaded
   Sensors: 325, Timesteps: 52116, Features: 12
   Train ends at timestep: 36481
   Val ends at timestep:   41692


Cell 2 — Define evaluation metrics

In [ ]:
# These three metrics are the standard for traffic prediction papers
# You'll use them to compare ALL models on the same scale

def mae(y_true, y_pred):
    """Mean Absolute Error — average mph error"""
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    """Root Mean Square Error — penalizes large errors more"""
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mape(y_true, y_pred, eps=1e-5):
    """Mean Absolute Percentage Error — relative error"""
    mask = np.abs(y_true) > eps   # avoid division by zero at standstills
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluate(y_true, y_pred, label=""):
    """Print all three metrics at once"""
    m  = mae(y_true, y_pred)
    r  = rmse(y_true, y_pred)
    p  = mape(y_true, y_pred)
    print(f"  {label:30s}  MAE={m:.4f}  RMSE={r:.4f}  MAPE={p:.2f}%")
    return {'MAE': m, 'RMSE': r, 'MAPE': p}

# ⚠️ IMPORTANT: metrics are on SCALED data (not real mph yet)
# At the end we'll convert back to mph using the scaler
print("✅ Metrics defined: MAE, RMSE, MAPE")

✅ Metrics defined: MAE, RMSE, MAPE


Cell 3 — Recreate the Dataset and DataLoader

In [ ]:
# Same TrafficDataset class from the data loader notebook
# We need to redefine it here since this is a new notebook

class TrafficDataset(Dataset):
    def __init__(self, data_3d, split_indices,
                 input_steps=12, pred_steps=12):
        self.data = data_3d
        self.indices = split_indices
        self.input_steps = input_steps
        self.pred_steps = pred_steps

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        t = self.indices[i]
        x = self.data[t - self.input_steps : t]           # (12, 325, 12)
        y = self.data[t : t + self.pred_steps, :, SPEED_IDX]  # (12, 325)
        return torch.tensor(x, dtype=torch.float32), \
               torch.tensor(y, dtype=torch.float32)

# Build index lists
train_indices = list(range(INPUT_STEPS, TRAIN_END - PRED_STEPS))
val_indices   = list(range(TRAIN_END + INPUT_STEPS, VAL_END - PRED_STEPS))
test_indices  = list(range(VAL_END + INPUT_STEPS, N_TIMES - PRED_STEPS))

train_dataset = TrafficDataset(data_3d, train_indices)
val_dataset   = TrafficDataset(data_3d, val_indices)
test_dataset  = TrafficDataset(data_3d, test_indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"✅ Datasets ready")
print(f"   Train: {len(train_dataset):,} samples")
print(f"   Val:   {len(val_dataset):,} samples")
print(f"   Test:  {len(test_dataset):,} samples")

✅ Datasets ready
   Train: 36,457 samples
   Val:   5,187 samples
   Test:  10,400 samples


Cell 4 — Baseline 1: Historical Average

In [ ]:
# ── MODEL 1: HISTORICAL AVERAGE ──────────────────────────────
# Idea: for each sensor, for each hour-of-day and day-of-week,
# compute the average speed in the training data.
# Prediction = that average. No learning, no gradients.

print("="*55)
print("BASELINE 1: Historical Average (HA)")
print("="*55)

# The 3D data speed slice for training: shape (TRAIN_END, 325)
train_speeds = data_3d[:TRAIN_END, :, SPEED_IDX]  # (36481, 325)

# Build hour and dayofweek arrays for training timesteps
# Feature index 1 = hour, index 2 = dayofweek (from FEATURE_COLS)
HOUR_IDX = 1
DOW_IDX  = 2

train_hours = data_3d[:TRAIN_END, 0, HOUR_IDX]   # same for all sensors
train_dows  = data_3d[:TRAIN_END, 0, DOW_IDX]

# Compute mean speed per (hour, dayofweek, sensor)
# Result: lookup table of shape (24, 7, 325)
ha_table = np.zeros((24, 7, N_SENSORS))
counts   = np.zeros((24, 7, N_SENSORS))

# Round to integers (they were normalized — decode hour/dow back)
# Actually: hour and dow are normalized too. Let's use raw values from df instead
# Simpler: use the position within the day directly from timestep index

# Each day has 288 timesteps (24*12). Hour = (timestep % 288) // 12
# dayofweek: day index = timestep // 288, dow = day_index % 7
for t in range(TRAIN_END):
    hour = (t % 288) // 12           # 0-23
    dow  = (t // 288) % 7            # 0-6
    ha_table[hour, dow, :] += train_speeds[t, :]
    counts[hour, dow, :]   += 1

# Avoid divide by zero
counts[counts == 0] = 1
ha_table /= counts

print(f"✅ Historical average table built: shape {ha_table.shape}")
print(f"   (24 hours × 7 days × 325 sensors)")

# Predict on TEST set
print("\nPredicting on test set...")
test_preds_ha = []
test_trues_ha = []

for t in test_indices[:3000]:   # use first 3000 test samples for speed
    hour = (t % 288) // 12
    dow  = (t // 288) % 7
    # Prediction: repeat the HA value for all pred_steps
    pred = np.tile(ha_table[hour, dow, :], (PRED_STEPS, 1))  # (12, 325)
    true = data_3d[t : t + PRED_STEPS, :, SPEED_IDX]         # (12, 325)
    test_preds_ha.append(pred)
    test_trues_ha.append(true)

test_preds_ha = np.array(test_preds_ha)  # (3000, 12, 325)
test_trues_ha = np.array(test_trues_ha)

print(f"\nResults on test set (scaled values):")
ha_results = {}
for h, label in [(2, '15 min'), (5, '30 min'), (11, '60 min')]:
    res = evaluate(
        test_trues_ha[:, h, :].flatten(),
        test_preds_ha[:, h, :].flatten(),
        label=f'HA @ {label}'
    )
    ha_results[label] = res

# Save HA table for later comparison
np.save(f'{DRIVE_FOLDER}/ha_table.npy', ha_table)
print(f"\n✅ Baseline 1 complete. HA table saved.")

BASELINE 1: Historical Average (HA)
✅ Historical average table built: shape (24, 7, 325)
   (24 hours × 7 days × 325 sensors)

Predicting on test set...

Results on test set (scaled values):
  HA @ 15 min                     MAE=0.3102  RMSE=0.6234  MAPE=184.55%
  HA @ 30 min                     MAE=0.3299  RMSE=0.6642  MAPE=195.55%
  HA @ 60 min                     MAE=0.3756  RMSE=0.7591  MAPE=217.65%

✅ Baseline 1 complete. HA table saved.


Cell 5 — Baseline 2: ARIMA

In [ ]:
# ── MODEL 2: ARIMA ────────────────────────────────────────────
# ARIMA is a classical statistical model for time series.
# It's too slow to run on all 325 sensors, so we sample 20 sensors,
# train ARIMA on each, and average the results.
# This is the standard approach in traffic research papers.

print("="*55)
print("BASELINE 2: ARIMA (sampled sensors)")
print("="*55)

!pip install -q statsmodels
from statsmodels.tsa.arima.model import ARIMA

# Sample 20 sensors evenly across the network
sample_sensor_indices = np.linspace(0, N_SENSORS-1, 20, dtype=int)
print(f"Running ARIMA on {len(sample_sensor_indices)} sampled sensors...")

arima_results_all = {label: [] for label in ['15 min', '30 min', '60 min']}

for i, s_idx in enumerate(sample_sensor_indices):
    # Training data for this sensor
    train_series = data_3d[:TRAIN_END, s_idx, SPEED_IDX]
    # Use last 2000 timesteps of training for speed (ARIMA is slow)
    train_series = train_series[-2000:]

    try:
        # ARIMA(1,1,1) is a standard starting configuration
        # p=1 (autoregressive), d=1 (differencing), q=1 (moving average)
        model = ARIMA(train_series, order=(1, 1, 1))
        fitted = model.fit()

        # Predict on a few test samples
        test_sample_indices = test_indices[:200]
        preds_sensor, trues_sensor = [], []

        for t in test_sample_indices:
            true_vals = data_3d[t:t+PRED_STEPS, s_idx, SPEED_IDX]
            # Forecast PRED_STEPS steps ahead
            forecast = fitted.forecast(steps=PRED_STEPS)
            preds_sensor.append(forecast)
            trues_sensor.append(true_vals)

        preds_sensor = np.array(preds_sensor)  # (200, 12)
        trues_sensor = np.array(trues_sensor)

        for h, label in [(2, '15 min'), (5, '30 min'), (11, '60 min')]:
            m = mae(trues_sensor[:, h], preds_sensor[:, h])
            arima_results_all[label].append(m)

        print(f"  Sensor {i+1:2d}/{len(sample_sensor_indices)} done")

    except Exception as e:
        print(f"  Sensor {i+1:2d} failed: {e}")

# Average across sensors
print(f"\nARIMA Results (averaged across {len(sample_sensor_indices)} sensors):")
arima_results = {}
for label in ['15 min', '30 min', '60 min']:
    avg_mae = np.mean(arima_results_all[label])
    print(f"  ARIMA @ {label:6s}  MAE={avg_mae:.4f}")
    arima_results[label] = {'MAE': avg_mae}

print(f"\n✅ Baseline 2 complete.")

BASELINE 2: ARIMA (sampled sensors)
Running ARIMA on 20 sampled sensors...
  Sensor  1/20 done
  Sensor  2/20 done
  Sensor  3/20 done
  Sensor  4/20 done
  Sensor  5/20 done
  Sensor  6/20 done
  Sensor  7/20 done
  Sensor  8/20 done
  Sensor  9/20 done
  Sensor 10/20 done
  Sensor 11/20 done
  Sensor 12/20 done
  Sensor 13/20 done
  Sensor 14/20 done
  Sensor 15/20 done
  Sensor 16/20 done
  Sensor 17/20 done
  Sensor 18/20 done
  Sensor 19/20 done
  Sensor 20/20 done

ARIMA Results (averaged across 20 sensors):
  ARIMA @ 15 min  MAE=0.3158
  ARIMA @ 30 min  MAE=0.3165
  ARIMA @ 60 min  MAE=0.3211

✅ Baseline 2 complete.


Cell 6 — Baseline 3: LSTM

In [ ]:
# ── MODEL 3: LSTM ─────────────────────────────────────────────
# Long Short-Term Memory — a recurrent neural network that reads
# a sequence of inputs and maintains a "memory" of past patterns.
# It will see 12 timesteps of all features and predict future speeds.
# No spatial awareness — treats each sensor independently.

print("="*55)
print("BASELINE 3: LSTM")
print("="*55)

class LSTMBaseline(nn.Module):
    def __init__(self, n_features, hidden_dim, n_layers, n_sensors, pred_steps):
        super().__init__()
        """
        n_features:  number of input features per sensor (12)
        hidden_dim:  size of LSTM hidden state (controls model capacity)
        n_layers:    how many LSTM layers to stack
        n_sensors:   325 (we flatten sensors into the batch dimension)
        pred_steps:  12 (how many future steps to predict)
        """
        self.n_sensors  = n_sensors
        self.pred_steps = pred_steps
        self.hidden_dim = hidden_dim

        # The LSTM reads (input_steps, n_features) per sensor
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,    # input shape: (batch, seq, features)
            dropout=0.1
        )

        # Output layer: maps hidden state → pred_steps predictions
        self.fc = nn.Linear(hidden_dim, pred_steps)

    def forward(self, x):
        """
        x shape: (batch, input_steps, n_sensors, n_features)
                  e.g.  (32,    12,       325,      12)

        The trick: treat each sensor in each batch as an independent sample.
        Reshape to (batch * n_sensors, input_steps, n_features)
        Run LSTM, then reshape back.
        """
        B, T, N, F = x.shape

        # Merge batch and sensor dimensions
        x = x.permute(0, 2, 1, 3)          # (B, N, T, F)
        x = x.reshape(B * N, T, F)         # (B*N, T, F)

        # Run through LSTM
        lstm_out, _ = self.lstm(x)          # (B*N, T, hidden_dim)
        last_hidden = lstm_out[:, -1, :]    # take last timestep: (B*N, hidden_dim)

        # Predict future steps
        out = self.fc(last_hidden)          # (B*N, pred_steps)

        # Reshape back to (B, N, pred_steps) then (B, pred_steps, N)
        out = out.reshape(B, N, self.pred_steps)
        out = out.permute(0, 2, 1)          # (B, pred_steps, N)
        return out


# Set up device (GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Instantiate model
lstm_model = LSTMBaseline(
    n_features=N_FEATURES,
    hidden_dim=64,
    n_layers=2,
    n_sensors=N_SENSORS,
    pred_steps=PRED_STEPS
).to(device)

# Count parameters
total_params = sum(p.numel() for p in lstm_model.parameters())
print(f"LSTM parameters: {total_params:,}")

# Optimizer and loss
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)
loss_fn   = nn.MSELoss()

print(f"\n✅ LSTM model ready")

BASELINE 3: LSTM
Using device: cpu
LSTM parameters: 54,028

✅ LSTM model ready


Cell 7 — Train the LSTM

In [ ]:
# Training loop
# For each epoch: feed batches → compute loss → update weights

N_EPOCHS = 10   # 10 epochs is enough for a baseline on Colab free tier

print(f"Training LSTM for {N_EPOCHS} epochs...")
print(f"(Each epoch ≈ 5-8 minutes on Colab Pro GPU)\n")

best_val_loss = float('inf')
train_losses, val_losses = [], []

for epoch in range(1, N_EPOCHS + 1):

    # ── TRAINING ──
    lstm_model.train()
    epoch_train_loss = 0
    n_batches = 0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)   # (32, 12, 325, 12)
        y_batch = y_batch.to(device)   # (32, 12, 325)

        optimizer.zero_grad()           # reset gradients
        pred = lstm_model(x_batch)      # forward pass: (32, 12, 325)
        loss = loss_fn(pred, y_batch)   # compute loss
        loss.backward()                 # backpropagation
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
        optimizer.step()                # update weights

        epoch_train_loss += loss.item()
        n_batches += 1

    avg_train_loss = epoch_train_loss / n_batches

    # ── VALIDATION ──
    lstm_model.eval()
    epoch_val_loss = 0
    n_val_batches  = 0

    with torch.no_grad():   # no gradients needed during validation
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            pred = lstm_model(x_batch)
            loss = loss_fn(pred, y_batch)
            epoch_val_loss += loss.item()
            n_val_batches  += 1

    avg_val_loss = epoch_val_loss / n_val_batches
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch:2d}/{N_EPOCHS}  "
          f"Train Loss: {avg_train_loss:.4f}  "
          f"Val Loss: {avg_val_loss:.4f}")

    # Save best model to Drive
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(lstm_model.state_dict(),
                   f'{DRIVE_FOLDER}/lstm_best.pt')
        print(f"            ✅ Best model saved (val loss: {best_val_loss:.4f})")

print(f"\n✅ Training complete. Best val loss: {best_val_loss:.4f}")

Training LSTM for 10 epochs...
(Each epoch ≈ 5-8 minutes on Colab Pro GPU)

Epoch  1/10  Train Loss: 0.2671  Val Loss: 0.2946
            ✅ Best model saved (val loss: 0.2946)
Epoch  2/10  Train Loss: 0.2063  Val Loss: 0.2405
            ✅ Best model saved (val loss: 0.2405)
Epoch  3/10  Train Loss: 0.1909  Val Loss: 0.2314
            ✅ Best model saved (val loss: 0.2314)
Epoch  4/10  Train Loss: 0.1850  Val Loss: 0.2297
            ✅ Best model saved (val loss: 0.2297)
Epoch  5/10  Train Loss: 0.1825  Val Loss: 0.2277
            ✅ Best model saved (val loss: 0.2277)
Epoch  6/10  Train Loss: 0.1805  Val Loss: 0.2260
            ✅ Best model saved (val loss: 0.2260)
Epoch  7/10  Train Loss: 0.1788  Val Loss: 0.2264
Epoch  8/10  Train Loss: 0.1773  Val Loss: 0.2263
Epoch  9/10  Train Loss: 0.1764  Val Loss: 0.2225
            ✅ Best model saved (val loss: 0.2225)
Epoch 10/10  Train Loss: 0.1751  Val Loss: 0.2211
            ✅ Best model saved (val loss: 0.2211)

✅ Training complete. Be

Cell 8 — Evaluate LSTM on test set

In [ ]:
# Load the best saved model
lstm_model.load_state_dict(
    torch.load(f'{DRIVE_FOLDER}/lstm_best.pt', map_location=device)
)
lstm_model.eval()

print("Evaluating LSTM on test set...")

all_preds, all_trues = [], []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        pred = lstm_model(x_batch)
        all_preds.append(pred.cpu().numpy())
        all_trues.append(y_batch.numpy())

all_preds = np.concatenate(all_preds, axis=0)  # (n_test, 12, 325)
all_trues = np.concatenate(all_trues, axis=0)

print(f"\nLSTM Results (scaled values):")
lstm_results = {}
for h, label in [(2, '15 min'), (5, '30 min'), (11, '60 min')]:
    res = evaluate(
        all_trues[:, h, :].flatten(),
        all_preds[:, h, :].flatten(),
        label=f'LSTM @ {label}'
    )
    lstm_results[label] = res

# Save predictions for later comparison
np.save(f'{DRIVE_FOLDER}/lstm_test_preds.npy', all_preds)
np.save(f'{DRIVE_FOLDER}/lstm_test_trues.npy', all_trues)
print(f"\n✅ LSTM predictions saved to Drive")

Evaluating LSTM on test set...

LSTM Results (scaled values):
  LSTM @ 15 min                   MAE=0.1563  RMSE=0.3186  MAPE=90.15%
  LSTM @ 30 min                   MAE=0.2091  RMSE=0.4389  MAPE=123.99%
  LSTM @ 60 min                   MAE=0.2761  RMSE=0.5575  MAPE=173.10%

✅ LSTM predictions saved to Drive


Cell 9 — Final comparison table

In [ ]:
# ── COMPARISON TABLE ──────────────────────────────────────────
# Show all baselines side by side at each prediction horizon
# This is the table you'll put in your thesis

print("\n" + "="*65)
print("BASELINE COMPARISON TABLE (scaled MAE)")
print("="*65)
print(f"{'Model':<20} {'15 min':>12} {'30 min':>12} {'60 min':>12}")
print("-"*65)

for label in ['15 min', '30 min', '60 min']:
    pass

models = {
    'Hist. Average': ha_results,
    'ARIMA':         arima_results,
    'LSTM':          lstm_results,
}

for model_name, results in models.items():
    row = f"{model_name:<20}"
    for horizon in ['15 min', '30 min', '60 min']:
        mae_val = results[horizon]['MAE']
        row += f"  {mae_val:>10.4f}"
    print(row)

print("="*65)
print("\n(Lower MAE = better. Your GNN+Transformer model should")
print(" beat these numbers at every horizon.)\n")

# Save all results to Drive
# float() converts numpy float32 → Python float, which JSON can handle
all_results = {
    'HA':    {k: float(v['MAE']) for k, v in ha_results.items()},
    'ARIMA': {k: float(v['MAE']) for k, v in arima_results.items()},
    'LSTM':  {k: float(v['MAE']) for k, v in lstm_results.items()},
}
with open(f'{DRIVE_FOLDER}/baseline_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print("✅ All baseline results saved to baseline_results.json")
print("✅ Phase 3 complete — ready for Phase 4 (GNN + Transformer)")


BASELINE COMPARISON TABLE (scaled MAE)
Model                      15 min       30 min       60 min
-----------------------------------------------------------------
Hist. Average             0.3102      0.3299      0.3756
ARIMA                     0.3158      0.3165      0.3211
LSTM                      0.1563      0.2091      0.2761

(Lower MAE = better. Your GNN+Transformer model should
 beat these numbers at every horizon.)

✅ All baseline results saved to baseline_results.json
✅ Phase 3 complete — ready for Phase 4 (GNN + Transformer)
